# Step 4 — Final dataset cleaning

> Apply post-merge consistency checks: drop empty parent repos, fix dtypes, remove rows with broken joins.

**Paper section:** Step 1 — Repository selection and workflow extraction (offline preparation).

## Inputs
| Path | Format | Source |
|---|---|---|
| `data/3_merged_dataset.csv` | CSV | Step 3 |

## Outputs
| Path | Format | Consumer |
|---|---|---|
| `data/4_final_dataset.csv` | CSV | Step 5 (schema refinement) |
| `data/step4_report.json` | JSON | Step 5 |



In [1]:
# --- Notebook bootstrap ---
# Idempotent: chdir to repo root + reset sys.argv for argparse.
# Works in Jupyter/JupyterLab/Antigravity/VS Code/plain python on Linux+macOS+Windows.
from dataset_utils.notebook_bootstrap import setup

setup()

print("Step 4 — starting: final post-merge cleaning...", flush=True)
"""Step 4: final cleaning and validation of merged dataset.

Performs final hygiene checks on the merged dataset:
- strips extra spaces from all text fields;
- normalizes null-like tokens;
- validates and normalizes numeric fields (both workflow and repo);
- normalizes dates to YYYY-MM-DD format;
- validates date consistency (created <= updated);
- removes rows with missing required fields;
- outputs a single clean CSV.
"""

import argparse
import json
from dataclasses import asdict, dataclass
from datetime import UTC, datetime, timezone
from pathlib import Path

import polars as pl

NULL_TOKENS = {"", "null", "none", "nan", "na", "n/a", "undefined", "-"}
DATE_FORMATS = [
    "%Y-%m-%d",
    "%Y-%m-%d %H:%M:%S",
    "%Y-%m-%dT%H:%M:%S",
    "%Y-%m-%dT%H:%M:%SZ",
]

@dataclass
class FinalStats:
    total_rows: int = 0
    rows_written: int = 0
    removed_missing_required: int = 0
    removed_invalid_numeric: int = 0
    removed_invalid_date: int = 0
    removed_inverted_dates: int = 0

def normalize_text(value: object) -> str:
    if value is None:
        return ""
    trimmed = str(value).strip()
    if trimmed.lower() in NULL_TOKENS:
        return ""
    return trimmed

def normalize_number(value: str) -> int | None:
    if value == "" or value is None:
        return None
    try:
        float_val = float(value)
    except (ValueError, TypeError):
        return None
    if float_val < 0:
        return None
    if float_val != int(float_val):
        return None
    return int(float_val)

def normalize_date(value: str) -> str | None:
    if value == "" or value is None:
        return None

    try:
        ts = int(value)
        # Force UTC so the same epoch value resolves to the same date regardless
        # of the host timezone — required for reproducibility across machines.
        parsed = datetime.fromtimestamp(ts, tz=UTC)
        return parsed.strftime("%Y-%m-%d")
    except (ValueError, OSError, OverflowError, TypeError):
        pass

    for fmt in DATE_FORMATS:
        try:
            parsed = datetime.strptime(value, fmt)
            return parsed.strftime("%Y-%m-%d")
        except ValueError:
            continue
    return None

def get_column_groups(fieldnames: list[str]) -> tuple[list[str], list[str], list[str]]:
    required_columns = ["repository", "file_path", "uid"]

    numeric_columns = [
        col
        for col in fieldnames
        if col
        in {
            "commits",
            "branches",
            "releases",
            "contributors",
            "stars",
            "issues",
            "pullrequests",
            "codelines",
            "size",
            "repo_commits",
            "repo_branches",
            "repo_releases",
            "repo_contributors",
            "repo_stars",
            "repo_issues",
            "repo_pullrequests",
            "repo_codelines",
            "repo_size",
        }
    ]

    date_columns = [
        col
        for col in fieldnames
        if col
        in {
            "created",
            "updated",
            "repo_created",
            "repo_updated",
            "authored_date",
            "committed_date",
        }
    ]

    return required_columns, numeric_columns, date_columns

def clean_merged_dataset(input_path: Path, output_path: Path) -> FinalStats:
    stats = FinalStats()

    from dataset_utils.integrity import assert_input_matches_report

    input_sha = assert_input_matches_report(
        input_path=input_path,
        prior_report_path=Path("data/step3_report.json"),
        label="step 4 input",
    )

    print("  reading input CSV (may take a moment for large files)...", flush=True)
    df = pl.read_csv(input_path, encoding="utf-8-sig")
    fieldnames = df.columns
    stats.total_rows = len(df)

    required_columns, numeric_columns, date_columns = get_column_groups(fieldnames)

    text_cols = [
        c for c in fieldnames
        if c not in numeric_columns + date_columns and df[c].dtype not in (pl.Boolean,)
    ]
    for col in text_cols:
        df = df.with_columns(
            pl.col(col).map_elements(normalize_text, return_dtype=pl.String).alias(col)
        )

    missing_mask = pl.lit(False)
    for col in required_columns:
        if col in df.columns:
            missing_mask = missing_mask | (pl.col(col).is_null() | (pl.col(col) == ""))
    stats.removed_missing_required = df.select(missing_mask.sum()).item()

    for col in numeric_columns:
        if col in df.columns:
            df = df.with_columns(
                pl.col(col).cast(pl.Utf8, strict=False)
                .map_elements(normalize_number, return_dtype=pl.Int64)
                .alias(col)
            )
            null_count = df.filter(pl.col(col).is_null()).height
            stats.removed_invalid_numeric += null_count

    for col in date_columns:
        if col in df.columns:
            df = df.with_columns(
                pl.col(col).cast(pl.Utf8, strict=False)
                .map_elements(normalize_date, return_dtype=pl.String)
                .alias(col)
            )
            null_count = df.filter(pl.col(col).is_null()).height
            stats.removed_invalid_date += null_count

    date_pairs = [
        ("created", "updated"),
        ("repo_created", "repo_updated"),
        ("authored_date", "committed_date"),
    ]

    inverted_mask = pl.lit(False)
    for created_col, updated_col in date_pairs:
        if created_col in df.columns and updated_col in df.columns:
            created_expr = pl.col(created_col).str.to_date("%Y-%m-%d", strict=False)
            updated_expr = pl.col(updated_col).str.to_date("%Y-%m-%d", strict=False)
            inverted_mask = inverted_mask | (updated_expr < created_expr).fill_null(False)

    stats.removed_inverted_dates = df.select(inverted_mask.sum()).item()

    df = df.filter(~missing_mask & ~inverted_mask)
    stats.rows_written = len(df)

    output_path.parent.mkdir(parents=True, exist_ok=True)
    print("  writing output...", flush=True)
    df.write_csv(output_path, include_header=True)

    from dataset_utils.hashing import compute_file_hash
    report = {
        **asdict(stats),
        "input_sha256": input_sha,
        "output_file": str(output_path),
        "output_sha256": compute_file_hash(output_path),
    }
    report_path = output_path.parent / "step4_report.json"
    with report_path.open("w") as f:
        json.dump(report, f, indent=2)

    return stats

def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Final cleaning and validation of merged dataset.")
    parser.add_argument(
        "--input",
        type=Path,
        default=Path("data/3_merged_dataset.csv"),
        help="Input merged CSV file path.",
    )
    parser.add_argument(
        "--output",
        type=Path,
        default=Path("data/4_final_dataset.csv"),
        help="Output final cleaned CSV file path.",
    )
    return parser.parse_args()

def main() -> None:
    args = parse_args()
    stats = clean_merged_dataset(args.input, args.output)

    print("Final cleaning finished.")
    print(f"Input: {args.input}")
    print(f"Output: {args.output}")
    print("-" * 40)
    print(f"Total rows read: {stats.total_rows}")
    print(f"Rows written: {stats.rows_written}")
    print("-" * 40)
    print(f"Removed missing required fields: {stats.removed_missing_required}")
    print(f"Nulled invalid numeric values (rows kept): {stats.removed_invalid_numeric}")
    print(f"Nulled invalid dates (rows kept): {stats.removed_invalid_date}")
    print(f"Removed inverted dates: {stats.removed_inverted_dates}")

main()


Step 4 — starting: final post-merge cleaning...
  reading input CSV (may take a moment for large files)...
  writing output...
Final cleaning finished.
Input: data/3_merged_dataset.csv
Output: data/4_final_dataset.csv
----------------------------------------
Total rows read: 253925
Rows written: 253921
----------------------------------------
Removed missing required fields: 0
Nulled invalid numeric values (rows kept): 0
Nulled invalid dates (rows kept): 0
Removed inverted dates: 4


---

**Next step:** [`05_refine_columns.ipynb`](./05_refine_columns.ipynb)
